In [ ]:
import os
import pandas as pd
from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.preprocessing import TransactionEncoder

if os.path.exists('/workspace/data'):
    DATA_DIR = '/workspace/data'
    WORKSPACE_DIR = '/workspace'
elif os.path.exists('../environment/data'):
    DATA_DIR = '../environment/data'
    WORKSPACE_DIR = '..'
elif os.path.exists('environment/data'):
    DATA_DIR = 'environment/data'
    WORKSPACE_DIR = '.'
else:
    DATA_DIR = 'data'
    WORKSPACE_DIR = '.'

In [ ]:
txn   = pd.read_csv(f'{DATA_DIR}/transactions.csv')
items = pd.read_csv(f'{DATA_DIR}/items.csv')

raw_row_count = len(txn)
print(f'raw_row_count = {raw_row_count}')

In [ ]:
# Count return transactions before filtering
return_rows_removed = int((txn['quantity'] < 0).sum())
print(f'return_rows_removed = {return_rows_removed}')

# Keep only positive-quantity rows (removes returns qty<0 and voided items qty=0)
clean = txn[txn['quantity'] > 0].copy()

In [ ]:
# Identify basket_ids containing promotional bundle transactions.
# Some baskets have both regular purchases and promo bundle items under the same
# basket_id. Because bundle placement influences what a customer adds alongside it,
# the entire basket is not organically assembled and must be excluded.
contaminated_ids = set(txn[txn['transaction_type'] == 'promo_bundle']['basket_id'].unique())
contaminated_baskets_removed = int(len(contaminated_ids))
print(f'contaminated_baskets_removed = {contaminated_baskets_removed}')

clean = clean[~clean['basket_id'].isin(contaminated_ids)]

In [ ]:
# Remove exact duplicate rows from POS double-logging
clean = clean.drop_duplicates()

In [ ]:
# Count split baskets (basket_id spans more than one transaction_id)
split_basket_count = int(
    clean.groupby('basket_id')['transaction_id'].nunique().gt(1).sum()
)
print(f'split_basket_count = {split_basket_count}')

In [ ]:
# Resolve canonical item ID to unify pre- and post-migration product identifiers
clean = clean.merge(
    items[['item_id', 'canonical_item_id']],
    on='item_id',
    how='left'
)
# Drop rows with no catalogue match (deprecated SKUs absent from items.csv)
clean = clean.dropna(subset=['canonical_item_id'])
clean['canonical_item_id'] = clean['canonical_item_id'].astype(int)

canonical_names = (
    items[items['item_id'] == items['canonical_item_id']]
    .set_index('canonical_item_id')['item_name']
)

In [ ]:
# Within-basket return netting:
# Some items were purchased and returned within the same basket_id (customer changed
# their mind at checkout). Filtering quantity < 0 removes the return row but leaves
# the original purchase as a ghost. To detect ghost purchases, net quantities must be
# computed from ALL transaction rows — including returns already filtered from clean.
#
# Strategy: join ALL original transactions with canonical_item_id, then compute net
# quantity per (basket_id, canonical_item_id). Keep only pairs where net qty > 0.
txn_canonical = txn.merge(
    items[['item_id', 'canonical_item_id']],
    on='item_id',
    how='left'
).dropna(subset=['canonical_item_id'])
txn_canonical['canonical_item_id'] = txn_canonical['canonical_item_id'].astype(int)

# Restrict to non-contaminated baskets to avoid contaminated-basket noise
txn_nc = txn_canonical[~txn_canonical['basket_id'].isin(contaminated_ids)]
net_qty = txn_nc.groupby(['basket_id', 'canonical_item_id'])['quantity'].sum()
valid_pairs = net_qty[net_qty > 0].reset_index()[['basket_id', 'canonical_item_id']]

# Remove any (basket_id, canonical_item_id) pairs where the net quantity is <= 0
clean = clean.merge(valid_pairs, on=['basket_id', 'canonical_item_id'])

In [ ]:
# Group by basket_id (not transaction_id) to handle split baskets
# Use sorted lists for deterministic TransactionEncoder column order
baskets = (
    clean.groupby('basket_id')['canonical_item_id']
    .apply(lambda x: sorted(set(x)))
    .reset_index()
)
baskets = baskets[baskets['canonical_item_id'].apply(len) >= 2]
valid_basket_count = int(len(baskets))
print(f'valid_basket_count = {valid_basket_count}')

In [ ]:
te     = TransactionEncoder()
matrix = te.fit_transform(baskets['canonical_item_id'].tolist())
df_enc = pd.DataFrame(matrix, columns=te.columns_)
df_enc = df_enc.rename(
    columns={cid: canonical_names.get(cid, str(cid)) for cid in te.columns_}
)

MIN_SUPPORT    = 0.03
MIN_CONFIDENCE = 0.30

freq = apriori(df_enc, min_support=MIN_SUPPORT, use_colnames=True)
try:
    rules = association_rules(
        freq, metric='confidence', min_threshold=MIN_CONFIDENCE,
        num_itemsets=len(freq)
    )
except TypeError:
    rules = association_rules(
        freq, metric='confidence', min_threshold=MIN_CONFIDENCE
    )

print(f'Total rules before filtering: {len(rules)}')

In [ ]:
rules = rules[
    (rules['antecedents'].apply(len) == 1) &
    (rules['consequents'].apply(len) == 1)
].copy()

rules['antecedent'] = rules['antecedents'].apply(lambda x: next(iter(x)))
rules['consequent']  = rules['consequents'].apply(lambda x: next(iter(x)))

rules = (
    rules[['antecedent', 'consequent', 'support', 'confidence', 'lift']]
    .sort_values(['lift', 'antecedent'], ascending=[False, True])
    .reset_index(drop=True)
)

print(f'Total rules after filtering: {len(rules)}')
print('\nTop 10 rules by lift:')
print(rules.head(10).to_string())

output_path = f'{WORKSPACE_DIR}/association_rules.csv'
rules.to_csv(output_path, index=False)
print(f'\nSaved {len(rules)} rules to {output_path}')